# 181. RLOO：为什么简单 REINFORCE 也能做好 LLM RLHF？

> **面试问题：Leave-One-Out baseline 怎样降低方差？response token loss、KL shaping、stale rollout 和零方差组怎样实现？**

## 先给结论

RLOO 对同一 prompt 采 G 个响应，用其他 G-1 个响应的平均 reward 作为当前样本 baseline；它不需要 value network，且 baseline 不包含自身 reward。最终用 response token log-prob 乘 sequence advantage，并结合 reference KL。简单不等于无状态：rollout policy、reward/verifier、mask 与 batch 分组必须严格版本化。

## 推荐回答主线

1. 按 prompt 分组计算 leave-one-out baseline，证明优势和为零并对 reward 平移不变。
2. 把 sequence advantage 广播到 response token，prompt/padding 不参与 policy gradient。
3. 加入 per-token KL shaping、importance ratio/clip 和 rollout age，处理 online policy 漂移。
4. 监控 reward 方差、有效组、KL、长度、成本与独立评测，不把 reward 上升当质量结论。

## 教学实现边界

代码使用合成 log-prob/reward 展示 estimator，不执行真实模型采样；PPO/RLOO 实现细节、KL estimator 与分布式聚合必须以固定训练 recipe 为准。

## 一手资料

- [Back to Basics: REINFORCE Style RLHF](https://arxiv.org/abs/2402.14740)
- [REINFORCE](https://link.springer.com/article/10.1007/BF00992696)
- [Training language models to follow instructions](https://arxiv.org/abs/2203.02155)


In [ ]:
import hashlib
import json
import math
import warnings
from dataclasses import asdict, dataclass

import numpy as np

# 屏蔽当前运行环境由 torch 间接触发的 pynvml 弃用告警，不隐藏算法告警。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)
import torch

# P 个 prompt、每个 G 个响应；reward 同时覆盖全同组与有区分度组。
torch.manual_seed(181)
P, G, T = 3, 4, 6
rewards = torch.tensor([[1.0, 0.0, 0.5, 1.5], [0.2, 0.2, 0.2, 0.2], [-1.0, 0.0, 1.0, 2.0]])
response_mask = torch.tensor([[[1, 1, 1, 0, 0, 0]] * G, [[1, 1, 1, 1, 0, 0]] * G, [[1, 1, 1, 1, 1, 0]] * G], dtype=torch.float32)

assert rewards.shape == (P, G)
assert response_mask.shape == (P, G, T)
assert G > 1


## 1. Leave-One-Out baseline：每个样本只看同 prompt 的其他响应

`b_i=(sum_j r_j-r_i)/(G-1)`，优势 `A_i=r_i-b_i`。它等价于 `G/(G-1)*(r_i-group_mean)`，因此每组优势和为零；全同 reward 组没有学习信号。


In [ ]:
def rloo_advantages(group_rewards):
    if not isinstance(group_rewards, torch.Tensor) or group_rewards.ndim != 2:
        raise TypeError("group_rewards 必须是二维 Tensor")
    if not group_rewards.is_floating_point() or group_rewards.shape[1] <= 1:
        raise ValueError("RLOO 需要每组至少两个浮点 reward")
    if not torch.isfinite(group_rewards).all():
        raise ValueError("reward 必须全部有限")
    total = group_rewards.sum(1, keepdim=True)
    baseline = (total - group_rewards) / (group_rewards.shape[1] - 1)
    return group_rewards - baseline, baseline

# 每组优势和为零，全同 reward 组优势全零，非法分组或 NaN 必须拒绝。
advantages, baselines = rloo_advantages(rewards)
assert torch.allclose(advantages.sum(1), torch.zeros(P))
assert torch.allclose(advantages[1], torch.zeros(G))
assert baselines.shape == rewards.shape
for invalid_rewards in (torch.tensor([[1.0]]), torch.tensor([[1.0, float("nan")]])):
    try:
        rloo_advantages(invalid_rewards); assert False
    except (TypeError, ValueError):
        assert True


## 2. 平移不变与 prompt-local：不能用全 batch 均值替代

给某个 prompt 全部 reward 加常数不应改变优势；如果用全 batch baseline，高 reward prompt 会系统性压制低 reward prompt，混入题目难度。RLOO 只比较同 prompt 候选。


In [ ]:
# 每个 prompt 加不同常数，RLOO 优势保持；全局中心化结果会变化。
shift = torch.tensor([[10.0], [-3.0], [5.0]])
shifted = rewards + shift
shifted_adv, _ = rloo_advantages(shifted)
global_centered = rewards - rewards.mean()
shifted_global = shifted - shifted.mean()
assert torch.allclose(advantages, shifted_adv, atol=1e-6)
assert not torch.allclose(global_centered, shifted_global)
assert torch.allclose(advantages, G / (G - 1) * (rewards - rewards.mean(1, keepdim=True)))


## 3. Token log-prob：sequence advantage 只广播到有效 response

rollout 已给定采样 token，因此训练取这些 token 的新/旧 log-prob。prompt token、padding、截断后的无效位置都 mask；每条响应是一个 sequence reward，但梯度作用到其所有有效 response token。


In [ ]:
# 构造新旧策略对已采样 token 的 log-prob，mask 外放极值测试隔离。
new_logp = torch.randn(P, G, T, requires_grad=True)
old_logp = (new_logp.detach() + 0.05 * torch.randn(P, G, T))
new_logp_safe = torch.where(response_mask.bool(), new_logp, torch.zeros_like(new_logp))
old_logp_safe = torch.where(response_mask.bool(), old_logp, torch.zeros_like(old_logp))
sequence_new = (new_logp_safe * response_mask).sum(-1)
sequence_old = (old_logp_safe * response_mask).sum(-1)
assert sequence_new.shape == (P, G)
assert torch.equal(new_logp_safe[response_mask == 0], torch.zeros_like(new_logp_safe[response_mask == 0]))
assert response_mask.sum(-1).min() > 0


## 4. RLOO policy loss：baseline 与 reward 必须 stop-gradient

最小形式是 `-A * sum_t logπ(y_t)`。若用 importance ratio，可在 sequence 或 token 层构造并裁剪；这里先展示 on-policy estimator，并确保 advantage 不反传进 reward 模型。


In [ ]:
def rloo_policy_loss(sequence_logp, advantage, raw_ratio=None, clipped_ratio=None):
    if sequence_logp.shape != advantage.shape:
        raise ValueError("sequence log-prob 与 advantage 形状必须一致")
    if not torch.isfinite(sequence_logp).all() or not torch.isfinite(advantage).all():
        raise ValueError("policy loss 输入必须有限")
    fixed_advantage = advantage.detach()
    if raw_ratio is None and clipped_ratio is None:
        return -(sequence_logp * fixed_advantage).mean()
    if raw_ratio is None or clipped_ratio is None:
        raise ValueError("importance ratio 与 clipped ratio 必须成对提供")
    if raw_ratio.shape != advantage.shape or clipped_ratio.shape != advantage.shape:
        raise ValueError("importance ratio 形状错误")
    if not torch.isfinite(raw_ratio).all() or not torch.isfinite(clipped_ratio).all():
        raise ValueError("importance ratio 必须有限")
    # PPO 风格 surrogate：ratio 承担 dπ/dlogπ，不能再额外乘一次 sequence_logp。
    return -torch.minimum(raw_ratio * fixed_advantage, clipped_ratio * fixed_advantage).mean()

# on-policy 梯度符号正确，mask 外 token 无梯度；reward/baseline 不参与反传。
policy_loss = rloo_policy_loss(sequence_new, advantages)
policy_loss.backward(retain_graph=True)
positive = advantages > 0
assert torch.all(new_logp.grad.sum(-1)[positive] < 0)
assert new_logp.grad[response_mask == 0].abs().sum() == 0
assert advantages.requires_grad is False


## 5. Per-token KL shaping：成本应与实际 response 长度对齐

常把 `-beta*(logπ-logπ_ref)` 作为每 token non-score reward，再与末端任务 reward 合成 return。长响应会累计更多 KL，因此要监控长度；reference log-prob 固定。


In [ ]:
def shaped_sequence_reward(task_reward, policy_token_logp, reference_token_logp, mask, beta):
    if task_reward.ndim != 2 or policy_token_logp.shape != reference_token_logp.shape or policy_token_logp.shape != mask.shape:
        raise ValueError("reward、log-prob 或 mask 形状错误")
    if tuple(policy_token_logp.shape[:2]) != tuple(task_reward.shape):
        raise ValueError("token 张量的分组维必须与 reward 对齐")
    if not isinstance(beta, (int, float)) or not math.isfinite(beta) or beta < 0:
        raise ValueError("KL beta 必须是有限非负数")
    tensors = (task_reward, policy_token_logp, reference_token_logp, mask)
    if not all(torch.isfinite(tensor).all() for tensor in tensors):
        raise ValueError("KL shaping 输入必须全部有限")
    if not torch.all((mask == 0) | (mask == 1)):
        raise ValueError("response mask 必须是 0/1")
    # rollout reward 使用冻结 behavior/reference log-prob；显式 detach，learner 更新不能改写 advantage。
    sampled_log_ratio = policy_token_logp.detach() - reference_token_logp.detach()
    token_kl_penalty = -beta * sampled_log_ratio * mask
    return task_reward.detach() + token_kl_penalty.sum(-1), token_kl_penalty

# shaped reward 由 behavior policy 固定，并立即进入 LOO advantage；beta=0 回到任务 reward。
reference_token_logp = old_logp_safe - 0.03
shaped, kl_tokens = shaped_sequence_reward(rewards, old_logp_safe, reference_token_logp, response_mask, 0.1)
shaped_advantages, shaped_baselines = rloo_advantages(shaped)
zero_beta, _ = shaped_sequence_reward(rewards, old_logp_safe, reference_token_logp, response_mask, 0.0)
assert torch.allclose(zero_beta, rewards)
assert torch.allclose(shaped_advantages.sum(1), torch.zeros(P), atol=1e-6)
assert kl_tokens[response_mask == 0].abs().sum() == 0
assert torch.isfinite(shaped_advantages).all()
try:
    shaped_sequence_reward(rewards, old_logp_safe, reference_token_logp, response_mask, float("nan")); assert False
except ValueError:
    assert True


## 6. Stale rollout：importance ratio 与 policy version 必须显式处理

生成和训练解耦时，数据来自旧 policy。ratio 过大意味着 estimator 高方差或越过 trust region；可限制 rollout age、裁剪 ratio 或丢弃，且日志保存行为策略 log-prob。


In [ ]:
def clipped_importance_ratio(new_sequence_logp, behavior_sequence_logp, clip=0.2, max_abs_log_ratio=30.0):
    if new_sequence_logp.shape != behavior_sequence_logp.shape:
        raise ValueError("新旧 sequence log-prob 形状必须一致")
    if not all(torch.isfinite(x).all() for x in (new_sequence_logp, behavior_sequence_logp)):
        raise ValueError("新旧 sequence log-prob 必须有限")
    if not all(isinstance(x, (int, float)) and math.isfinite(x) for x in (clip, max_abs_log_ratio)):
        raise ValueError("ratio 配置必须有限")
    if not 0 <= clip < 1 or max_abs_log_ratio <= 0:
        raise ValueError("clip 或 log-ratio 门禁非法")
    log_ratio = new_sequence_logp - behavior_sequence_logp
    if log_ratio.abs().max() > max_abs_log_ratio:
        raise ValueError("policy 漂移过大，拒绝可能溢出的 rollout")
    raw = torch.exp(log_ratio)
    return raw, raw.clamp(1 - clip, 1 + clip)

def rollout_acceptable(current_version, rollout_version, max_age):
    if any(type(value) is not int for value in (current_version, rollout_version, max_age)):
        raise TypeError("policy version 与 max_age 必须是整数")
    if min(current_version, rollout_version, max_age) < 0:
        raise ValueError("policy version 与 max_age 不得为负")
    return 0 <= current_version - rollout_version <= max_age

def rloo_training_objective(task_reward, policy_token_logp, behavior_token_logp, reference_token_logp, mask, beta, clip, current_version, rollout_version, max_age):
    if not rollout_acceptable(current_version, rollout_version, max_age):
        raise ValueError("rollout 已过期或来自未来 policy")
    # reward/advantage 固定在采样时的 behavior policy；learner 只通过 ratio/surrogate 接收梯度。
    shaped_reward, kl_penalty = shaped_sequence_reward(task_reward, behavior_token_logp, reference_token_logp, mask, beta)
    advantage, baseline = rloo_advantages(shaped_reward)
    sequence_policy = (policy_token_logp * mask).sum(-1)
    sequence_behavior = (behavior_token_logp.detach() * mask).sum(-1)
    raw_ratio, clipped_ratio = clipped_importance_ratio(sequence_policy, sequence_behavior, clip)
    loss = rloo_policy_loss(sequence_policy, advantage, raw_ratio, clipped_ratio)
    return {"loss": loss, "reward": shaped_reward, "advantage": advantage, "baseline": baseline, "kl_tokens": kl_penalty, "raw_ratio": raw_ratio, "clipped_ratio": clipped_ratio}

# KL、importance ratio 与版本门禁进入同一目标；改变 learner 不得改写 rollout reward/advantage。
training = rloo_training_objective(rewards, new_logp_safe, old_logp_safe, reference_token_logp, response_mask, 0.1, 0.2, 10, 8, 2)
changed_learner = new_logp_safe + 0.01 * response_mask
changed_training = rloo_training_objective(rewards, changed_learner, old_logp_safe, reference_token_logp, response_mask, 0.1, 0.2, 10, 8, 2)
assert torch.isfinite(training["loss"])
assert torch.allclose(training["advantage"], shaped_advantages)
assert torch.equal(training["reward"], changed_training["reward"])
assert torch.equal(training["advantage"], changed_training["advantage"])
assert not torch.allclose(training["raw_ratio"], changed_training["raw_ratio"])
assert ((training["clipped_ratio"] >= 0.8) & (training["clipped_ratio"] <= 1.2)).all()
objective_gradient = torch.autograd.grad(training["loss"], new_logp, retain_graph=True)[0]
assert torch.isfinite(objective_gradient).all() and objective_gradient.abs().sum() > 0
assert objective_gradient[response_mask == 0].abs().sum() == 0
for bad_versions in ((10, 7, 2), (10, 11, 2)):
    try:
        rloo_training_objective(rewards, new_logp_safe, old_logp_safe, reference_token_logp, response_mask, 0.1, 0.2, *bad_versions); assert False
    except ValueError:
        assert True
for bad_ratio_args in ((torch.tensor([[100.0]]), torch.zeros(1, 1), 0.2), (torch.zeros(1, 1), torch.zeros(1, 1), -0.1)):
    try:
        clipped_importance_ratio(*bad_ratio_args); assert False
    except ValueError:
        assert True
try:
    rollout_acceptable(1, 0, -1); assert False
except ValueError:
    assert True


## 7. 有效学习组：全同 reward、解析失败与 reward 饱和要分开报告

组内零方差不会产生 RLOO 信号；可能因为任务太易/太难、verifier 崩溃或 reward 被截平。面板应报告有效组比例、组内标准差、成功率、parse/verifier error 与长度。


In [ ]:
def group_signal_report(group_rewards, eps=1e-8):
    if not isinstance(eps, (int, float)) or not math.isfinite(eps) or eps < 0:
        raise ValueError("eps 必须有限非负")
    rloo_advantages(group_rewards)
    std = group_rewards.std(1, unbiased=False)
    return {
        "informative_fraction": float((std > eps).float().mean()),
        "mean_group_std": float(std.mean()),
        "zero_groups": int((std <= eps).sum()),
    }

# 面板读取真正用于训练的 shaped rollout reward，而不是脱离主路径的 raw task reward。
report = group_signal_report(training["reward"].detach())
assert report["zero_groups"] == 1
assert math.isclose(report["informative_fraction"], 2 / 3, abs_tol=1e-6)
assert report["mean_group_std"] >= 0


## 8. 制品与发布：rollout、reward、reference 和 learner 是四个版本

保存 prompt id、response tokens/mask、behavior log-prob、policy/ref/reward/verifier hash、采样参数和时间。发布用独立 evaluator、人评、安全 slice、KL、长度、pass@1 与每有效组成本验收。


In [ ]:
@dataclass(frozen=True)
class RLOOArtifact:
    learner_policy: str
    behavior_policy: str
    reference: str
    reward: str
    verifier: str
    group_size: int
    kl_beta: float
    ratio_clip: float
    max_rollout_age: int

def artifact_hash(artifact):
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()

# learner/behavior/reference 四类版本与目标配置进入摘要，改变 reward 产生不同制品。
artifact = RLOOArtifact("policy-v10", "policy-v8", "ref-v1", "rm-v5", "verify-v3", G, 0.1, 0.2, 2)
digest = artifact_hash(artifact)
assert artifact.group_size == rewards.shape[1]
assert artifact.learner_policy != artifact.behavior_policy
assert len(digest) == 64
assert digest != artifact_hash(RLOOArtifact("policy-v10", "policy-v8", "ref-v1", "rm-v6", "verify-v3", G, 0.1, 0.2, 2))


## 面试收束：Agent/RAG 的算法只是控制面的一部分

推荐回答顺序是：任务目标和失败代价、状态/事件/证据合同、决策公式、可执行反例、离线与在线指标、权限和版本。受控环境只能证明状态机和数值关系，不能冒充开放网络、真实用户或真实模型结果。生产系统还要处理并发、超时、幂等、恶意内容、隐私、审计、灰度与回滚。

遇到追问时，主动区分模型判断与确定 verifier、计划与真实副作用、原始 observation 与 belief/memory、召回质量与生成归因，以及多尝试成功率与单次可靠性。
